# Solutions 4 — APIs, Observability, and Transactions

Six worked references on safe public contracts, pagination, service metrics, SQL idempotency, resource lifetime, and lifecycle design.

**Level:** Advanced · **Study mode:** Worked reference examples · **Prerequisites:** Relevant lessons and exercise workbook.

These solutions show one defensible approach each; multiple implementations may be correct. Review the contracts, test boundaries, and discuss design trade-offs before comparing line-by-line.

## 1. Structured error envelope

Use stable error codes and correlation IDs while avoiding internal implementation details. Clients should not parse English messages.

In [ ]:
def make_error(code, message, request_id):
    if not code or not request_id:
        raise ValueError("missing error metadata")
    return {"error": {"code": code, "message": message, "request_id": request_id}}

error = make_error("INVALID_INPUT", "Missing SKU", "request-100")
assert error["error"]["code"] == "INVALID_INPUT"
print(error)

**Extension:** How do you define stable error code semantics across API versions?

## 2. Pagination loop guard

Bound the number of pages and reject repeated cursors. Tokens are opaque; do not attempt to decode their internal format in the client.

In [ ]:
PAGES = {
    "one": {"items": [1, 2], "next": "two"},
    "two": {"items": [3], "next": None}
}
def get_all(fetch, cursor="one", max_pages=10):
    seen, result = set(), []
    while cursor is not None:
        if cursor in seen or len(seen) >= max_pages:
            raise RuntimeError("invalid pagination")
        seen.add(cursor)
        page = fetch(cursor)
        result.extend(page["items"])
        cursor = page["next"]
    return result

assert get_all(PAGES.__getitem__) == [1, 2, 3]

**Extension:** Why can offset pagination skip or repeat records during concurrent inserts?

## 3. Latency and availability summaries

State a measurement window, the request counts, and a precise percentile convention. Reject empty lists rather than inventing performance observations.

In [ ]:
import math
def nearest_rank(values, p):
    if not values or not 0 < p <= 100:
        raise ValueError("invalid percentile")
    ordered = sorted(values)
    return ordered[math.ceil(p / 100 * len(ordered)) - 1]

latencies = [10, 20, 30, 40, 50]
assert nearest_rank(latencies, 80) == 40
success, total = 996, 1000
assert success / total == 0.996
print("availability:", success / total)

**Extension:** Why should high-percentile latency be observed for a representative sample size?

## 4. Durable idempotency keys

SQLite uniqueness plus a transaction can make repeated identical requests return the same stored result. Conflicting requests with the same key must be rejected.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE outcomes (key TEXT PRIMARY KEY, payload TEXT NOT NULL, result TEXT NOT NULL)")
    def process(key, payload):
        with db:
            db.execute("INSERT OR IGNORE INTO outcomes VALUES (?, ?, ?)", (key, payload, "done"))
            old_payload, result = db.execute("SELECT payload, result FROM outcomes WHERE key=?", (key,)).fetchone()
            if old_payload != payload:
                raise ValueError("idempotency key conflict")
            return result
    assert process("k1", "amount:10") == process("k1", "amount:10")
finally:
    db.close()

**Extension:** How would this pattern integrate with a payment's external side effect safely?

## 5. Versioned API serialization

An old response remains unchanged when an optional field is introduced in v2. Public output should be constructed explicitly instead of dumping internal object dictionaries.

In [ ]:
def serialize(product, version=1):
    result = {"sku": product["sku"], "name": product["name"]}
    if version == 1:
        return result
    if version == 2:
        return {**result, "category": product.get("category")}
    raise ValueError("unsupported version")

item = {"sku": "P1", "name": "Pen", "category": "office"}
assert "category" not in serialize(item, 1)
assert serialize(item, 2)["category"] == "office"

**Extension:** When should a new API version be introduced rather than adding an optional field?

## 6. Resource lifecycle

Context managers guarantee cleanup even when the body raises. Explicit resource ownership is essential for threads, files, connections, and asynchronous tasks.

In [ ]:
from contextlib import contextmanager
events = []
@contextmanager
def managed():
    events.append("open")
    try:
        yield
    finally:
        events.append("close")
try:
    with managed():
        raise ValueError("simulated failure")
except ValueError:
    pass
assert events == ["open", "close"]

**Extension:** What additional steps would an async service need for graceful shutdown?

## Recap

- Reference solutions illustrate testing and contracts rather than exhaustive answers.
- Explain one alternative design for each pattern.
- Re-run every code cell in a fresh kernel before copying into a project.

**Next:** Work independently on the unsolved questions in the corresponding workbook.